<a href="https://colab.research.google.com/github/adhishagc/quantum-inspired-genetic-algorithm-qiskit-implementation-for-tsp/blob/master/project_01_and_02_python_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
#installing Quantum Kit
pip install qiskit

In [0]:
import numpy as np
import pandas as pd
%matplotlib inline

#quantum libraries
from qiskit import *
from qiskit import IBMQ
from qiskit.providers.ibmq import least_busy
from qiskit.tools.monitor import job_monitor
from qiskit.visualization import plot_histogram


In [19]:
token = '19d63723cbc1c4747be173678dc6132c1ae1f6fe269298e740c6d3a928379c57020a6f2d04a1607232a3a0968c40a09fce7ec6d1769d878169a70b04feb73d04'
IBMQ.save_account(token)
provider = IBMQ.load_account()

/usr/local/lib/python3.6/dist-packages/qiskit/providers/ibmq/credentials/configrc.py:130: UserWarning: Credentials already present. Set overwrite=True to overwrite.
  warnings.warn('Credentials already present. '
/usr/local/lib/python3.6/dist-packages/qiskit/providers/ibmq/ibmqfactory.py:181: UserWarning: Credentials are already in use. The existing account in the session will be replaced.
  warnings.warn('Credentials are already in use. The existing '


In [20]:
#Access to google drive. The Video Store dataset was uploaded to google drive.
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [0]:
#The dataset is load dataset to a dataframe
df = pd.read_csv('/content/gdrive/My Drive/Final Year Project/datasets/dist3.csv')

In [22]:
df.iloc[0][1]

5

In [0]:
class qubit:
  #a and b are alpha and beta representation of quantum superposition probabilities
  #superposition values are generated in this case using equiprobable random values
  def __init__(self):
    self.val = -1
    self.a = np.random.uniform(0,1)
    self.b = 1 - self.a
    
  def state(self):
    if(self.a>=self.b):
      print('0')
    else:
      print('1')
  
  def measure(self):
    if(self.a>=self.b):
      self.val = 0
    else:
      self.val = 1
    

     

In [0]:
def generateRandomSolution():
  d = np.empty([n_cities,n_cities])
  count = 0
  for i in range(n_cities):
    for j in range(n_cities):
      d[i][j] = count
      count=count+1
  return d

In [0]:
def createPopulation(popSize):
  for i in range(popSize):
    solutionSpace.append(generateRandomSolution())

In [0]:
#Crossover operation
def crossover():
  cross_col = 0
  
  for i in range(popSize-1):
    for k in range(i+1,popSize):
      parent_1 = solutionSpace[i]
      parent_2 = solutionSpace[k]

      for j in range(n_cities):
        temp = parent_1[j][cross_col]
        parent_1[j][cross_col] = parent_2[j][cross_col]
        parent_2[j][cross_col] = temp
        
        solutionSpace.append(parent_1)
        solutionSpace.append(parent_2)
  

In [0]:
#mutation
def permute(permute_thres,permute_prob):
  
  for chrom in range(len(solutionSpace)):
    #probability
    r = np.random.uniform(0,1)
    
    if(r <= permute_thres):
      #decided to apply permutation
      for i in range(n_cities):
        r = np.random.uniform(0,1)
        if(r<= permute_prob):
          j = np.random.randint(0,n_cities)
          
          if(i != j):
            #2 selected rows not the same
            sol = solutionSpace[chrom]
            
            for col in range(n_cities):
              temp = sol[i][col]
              sol[i][col] = sol[j][col]
              sol[j][col] = temp
            
            solutionSpace[chrom] = sol

In [0]:
#convert to one
def measure():
  for chrom in range(len(solutionSpace)):
    for i in range(n_cities):
      for j in range(n_cities):
        (solutionSpace[chrom])[i][j].measure()

In [0]:
#select 1 per row and 1 per col item
def checkItem():
  for chrom in range(len(solutionSpace)):
    row_counter = 0
    col_counter = 0
    addThis = False
    
    while row_counter <2 and col_counter <2:
      #check col wise
      for i in range(n_cities):
        for j in range(n_cities):
          if solutionSpace[chrom][i][j].val == 1:
            col_counter = col_counter + 1
            
          elif col_counter > 1:
            i = n_cities +1
            row_counter = 2
            break
          
      if col_counter <2 and row_counter <2:
        for i in range(n_cities):
          for j in range(n_cities):
            if solutionSpace[chrom][j][i].val == 1:
              row_counter = row_counter + 1
            
            elif row_counter > 1:
              i = n_cities +1
              break
              
      if col_counter <2 and row_counter <2:   
        addThis = True
        col_counter = 3
        row_counter = 3
  
    if(addThis != False):
      items.append(chrom)
        

In [0]:
def getDistance():
  for k in range(len(items)):
    visit_order = []
    dist = 0
    for i in range(n_cities):
      for j in range(n_cities):
        if solutionSpace[items[k]][i][j] == 1:
          visit_order.append(j)
    
    for i in range(n_cities):
      loc = visit_order[i]
      dist = dist + dis[i][loc]   
    
    distances[k][0] = item[k]
    distances[k][1] = dist
    

In [0]:
def quantum_getDistance():
  #updating the getDistance function for the simulation with qiskit
  for k in range(popSize):
    visit_order = []
    dist = 0
    for i in range(n_cities):
      for j in range(n_cities):
        if solutionSpace[k][i][j] == 1:
          visit_order.append(j)
          break #this break is a temporaly solution to have one 1 per row, there is a chance all cities are not visited. #FIX
          
    for i in range(len(visit_order)):
      loc = visit_order[i]
      dist = dist + df.iloc[i][loc]   
    
    distances[k][0] = k
    distances[k][1] = dist

In [32]:
def iterate(n):
  
  lowest_dist = -1
  lowest_dist_id = -1
  for i in range(n+1):
    
    #create quantum intereference
    #NEED TO Add a FUNCTION

    #crossover operation
    crossover()

    #permutate
    permute_thres = 0.5
    permute_prob = 0.6
    permute(permute_thres,permute_prob)

    #SKIPPING RANDOM SHIFTS

    #measuring qubits
    #measure()

    #get items which are coherent
    items = []
    checkItem()

    #distance measure
    distances = np.empty([len(items),2])
    getDistance()
  distance_df = pd.DataFrame(distances,columns=['chrom', 'distance'])
    
  #sorting the dataframe
  distance_df.sort_values(by=['distance'],inplace=True)
  distance_df = distance_df.reset_index(drop=True)
    
  #select top 4 , in the paper it was top 3 and 1 random
  if len(items)>3:
    #select the top 4
    for i in range(4):
      solutionSpace[i]
  else:
    #use this as the new population
  

SyntaxError: ignored

In [0]:
def create_quantum_circuit(size):
  # Create a Quantum Register with given size qubits.
  qr = QuantumRegister(size, 'qr')
  # create classical register
  cr = ClassicalRegister(size, 'cr')
  
  # Create a Quantum Circuit acting on the qr quantum register cr classical register
  circ = QuantumCircuit(qr,cr)
  
  return qr,cr,circ

In [0]:
def create_quantum_population(popSize,qubits):
  for i in range(popSize):
    qr,cr,circ = create_quantum_circuit(qubits)
    qr_list.append(qr)
    cr_list.append(cr)
    circ_list.append(circ)

In [0]:
def add_H_gate(circ_list):
  for i in range(len(circ_list)):
    for j in range(qubits):
      circ_list[i].h(qr_list[i][j])

In [0]:
def add_T_gate(circ_list):
  for i in range(len(circ_list)):
    for j in range(qubits):
      circ_list[i].t(qr_list[i][j])

In [0]:
def circ_measure(qr_list,cr_list,circ_list,results_list,backend_type):
  #shots_no = 1
  for i in range(len(circ_list)):
    circ_list[i].measure(qr_list[i],cr_list[i])
    
    backends = provider.backends() #get a list of available backends
    
    result = execute(circ_list[i], backend=backends[backend_type], shots=1, memory=True).result()
    memory = result.get_memory(circ_list[i])
    results_list.append(memory[0]) # it is fine to use the 0 index since only 1 shot is used.

In [0]:
def merge_solutions_array():
  for k in range(popSize):
    for i in range(n_cities):
      for j in range(n_cities):
        list_measurement_index = int(solutionSpace[k][i][j])
        solutionSpace[k][i][j] = list(results_list[k])[list_measurement_index]

In [0]:
def distance_sorting(distances):
  distance_df = pd.DataFrame(distances,columns=['chrom', 'distance'])
  #sorting the dataframe
  distance_df.sort_values(by=['distance'],inplace=True)
  distance_df = distance_df.reset_index(drop=True)
  
  return distance_df
  

In [0]:
def get_visiting_order(best_id):
  #visiting order
  order = []
  for i in range(n_cities):
    for j in range(n_cities):
      if solutionSpace[best_id][i][j] == 1:
        order.append(j)
        break
      #THIS IS WRONG. Just doing it to make the Structure. Issue is because of the INCOHERENT Solutions/duplicates
      elif j == n_cities-1 and solutionSpace[best_id][i][j] == 0:
        order.append(n_cities-1)      
        break
  
  return order
   

In [0]:
def send_job_to_ibm(shots,max_credits,circ_list,backend_type):
  backends = provider.backends()
  job_exp = execute(circ_list[0], backend=backends[backend_type], shots=shots, max_credits=max_credits)
  job_monitor(job_exp)
  
  return job_exp

In [0]:
def available_backends():
  #print("Available backends:")
  provider.backends()

In [0]:
def least_busy_backend():
  large_enough_devices = provider.backends(filters=lambda x: x.configuration().n_qubits < 10 and
                                                           not x.configuration().simulator)
  backend = least_busy(large_enough_devices)
  print("The best backend is " + backend.name())

In [0]:
#Obselete
def real_quantum_iterate(n,backend_type):
  add_H_gate(circ_list)
  circ_measure(qr_list,cr_list,circ_list,results_list)
  
  shots = 1024           # Number of shots to run the program (experiment); maximum is 8192 shots.
  max_credits = 3        # Maximum number of credits to spend on executions. 

  job_exp = send_job_to_ibm(shots,max_credits,circ_list,backend_type)
  result_exp = job_exp.result()
  counts_exp = result_exp.get_counts(circ_list[0])
  return counts_exp
  #plot_histogram([counts_exp])

In [0]:
def quantum_iterate(n,backend_type):
  add_H_gate(circ_list)
  for i in range(n):    
    #create quantum intereference
    add_T_gate(circ_list)
      

    #crossover operation
    crossover()

    #permutate
    permute_thres = 0.5
    permute_prob = 0.6
    permute(permute_thres,permute_prob)

  circ_measure(qr_list,cr_list,circ_list,results_list,backend_type)
  merge_solutions_array()
  quantum_getDistance()

In [0]:
n_cities = 4

#create population
popSize = 4
solutionSpace = []

#create population
createPopulation(popSize)

# Select the QasmSimulator from the Aer provider
#simulator = Aer.get_backend('qasm_simulator')

qr_list = []
cr_list = []
circ_list = []
results_list = []
distances = np.empty([popSize,2])
qubits = n_cities*n_cities

#type of backend
backend_type = 0 #0 - simulator 1 - 4 qubit 2 - 2 qubit 3 - 16 qubit

In [0]:
#create quantum circuites / population
create_quantum_population(popSize,qubits)

In [0]:
quantum_iterate(1,backend_type)

In [0]:
distance_df = distance_sorting(distances)

In [236]:
distance_df


,chrom,distance
0,3.0,6.0
1,1.0,17.0
2,2.0,18.0
3,0.0,21.0


In [0]:
#solution id of the best generated 
best_id = int(distance_df.loc[0]['chrom'])

In [0]:
order = get_visiting_order(best_id)

In [239]:
order

[0, 0, 3, 2]

In [240]:
circ_list[best_id].draw()